In [2]:
#!pip install optuna tqdm catboost scikit-learn tensorflow pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [3]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np
import datetime
from tqdm import tqdm

# ML Модели
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras (Все Нейросети на CUDA)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Метрики
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --- НАСТРОЙКА CUDA (GPU) ДЛЯ TENSORFLOW ---
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print(f"CUDA обнаружена! Доступно GPU: {len(physical_devices)}")
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
    except Exception as e:
        print(f"Ошибка настройки памяти GPU: {e}")
else:
    print("CUDA не обнаружена. Все модели будут использовать CPU.")

# --- Конфигурация ---
M = 30
SEQ_LENGTH = 30 # Окно для LSTM и Transformer
EMA_FAST = 33
EMA_SLOW = 100
FREQ_MS = 60000
MIN_CHUNK_LENGTH = 100
ZIP_FILE = 'dataset_rework.zip'
OUTPUT_DIR = 'dataset_flattened'

# --- 1. Вспомогательные функции (Распаковка и Чанки) ---
def unpack_dataset(zip_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        files = zip_ref.infolist()
        print(f"\nРаспаковка архива {zip_path}...")
        for file_info in tqdm(files, desc="Распаковка"):
            original_path = file_info.filename
            if file_info.is_dir() or original_path.endswith('/') or '__MACOSX' in original_path:
                continue
            flat_filename = original_path.replace('/', '_').replace('\\', '_')
            dest_path = os.path.join(output_dir, flat_filename)
            if not os.path.exists(dest_path):
                with zip_ref.open(file_info) as source_file, open(dest_path, 'wb') as target_file:
                    target_file.write(source_file.read())

def get_contiguous_chunks(df, freq_ms=60000):
    df['symbol'] = df['symbol'].astype(str)
    df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    df['diff'] = df.groupby('symbol')['timestamp'].diff()
    df['is_new_chunk'] = (df['diff'] != freq_ms) | (df['symbol'] != df['symbol'].shift(1))
    df['chunk_id'] = df.groupby('symbol')['is_new_chunk'].cumsum()
    df['chunk_full_id'] = df['symbol'] + "_" + df['chunk_id'].astype(str)
    return df

def filter_and_report_chunks(df, min_len=100):
    chunk_stats = df.groupby('chunk_full_id').size().reset_index(name='length')
    valid_chunk_ids = chunk_stats[chunk_stats['length'] >= min_len]['chunk_full_id'].tolist()
    return df[df['chunk_full_id'].isin(valid_chunk_ids)].copy()

# --- 2. Инжиниринг Плоских Признаков ---
def create_financial_features(df, m_window=30, ema_fast=33, ema_slow=100):
    df = df.copy()
    grouped = df.groupby('chunk_full_id')

    # БАЗОВЫЕ ПРИЗНАКИ (Без лагов. Именно они пойдут в LSTM и Transformer)
    df['log_return'] = grouped['close'].transform(lambda x: np.log(x / x.shift(1)))
    df['hl_spread'] = (df['high'] - df['low']) / df['close']
    df['body'] = (df['close'] - df['open']) / df['close']
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['close']
    df[f'dist_from_sma_{m_window}'] = grouped['close'].transform(lambda x: (x / x.rolling(m_window).mean()) - 1)

    grouped_new = df.groupby('chunk_full_id')
    df[f'volatility_{m_window}'] = grouped_new['log_return'].transform(lambda x: x.rolling(m_window).std())
    df['log_vol_change'] = grouped_new['volume'].transform(lambda x: np.log((x + 1) / (x.shift(1) + 1)))
    df[f'vol_vs_sma_{m_window}'] = grouped_new['volume'].transform(lambda x: x / (x.rolling(m_window).mean() + 1))

    # Сигналы rd_value
    df['rd_value_scaled'] = grouped_new['rd_value'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
    df['rd_value_diff'] = grouped_new['rd_value_scaled'].transform(lambda x: x.diff())
    df['rd_ema_fast'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_fast, adjust=False).mean())
    df['rd_ema_slow'] = grouped_new['rd_value'].transform(lambda x: x.ewm(span=ema_slow, adjust=False).mean())
    df['rd_ema_diff'] = df['rd_ema_fast'] - df['rd_ema_slow']
    df['rd_ema_cross_signal'] = np.where(df['rd_ema_fast'] > df['rd_ema_slow'], 1, -1)

    # Список строго базовых признаков (Без rd_value и без лагов) - для 3D сетей
    base_3d_features = [
        'log_return', 'hl_spread', 'body', 'upper_shadow', 'lower_shadow',
        f'dist_from_sma_{m_window}', f'volatility_{m_window}', 'log_vol_change', f'vol_vs_sma_{m_window}'
    ]

    # ГЕНЕРАЦИЯ ПЛОСКИХ ЛАГОВ (Только для CatBoost и MLP)
    lag_targets = ['log_return', 'hl_spread', 'body']
    lag_feats = {}
    print("Генерация исторических лагов (для 2D моделей)...")
    for col in tqdm(lag_targets, desc="Лаги"):
        for i in range(1, m_window + 1):
            lag_feats[f"{col}_lag_{i}"] = grouped_new[col].shift(i)

    df = pd.concat([df, pd.DataFrame(lag_feats)], axis=1)

    # Собираем все фичи для 2D таблиц
    all_features = base_3d_features + ['rd_value_scaled', 'rd_value_diff', 'rd_ema_fast', 'rd_ema_slow', 'rd_ema_diff', 'rd_ema_cross_signal'] + list(lag_feats.keys())

    df = df.dropna(subset=all_features).reset_index(drop=True)
    return df, sorted(all_features), base_3d_features

# --- 3. Правильная Трансформация в 3D (Исправлен Target Alignment) ---
def create_3d_sequences(df, feature_cols, seq_length=30):
    """Нарезает 3D окна. Внимание: в feature_cols подаются ТОЛЬКО базовые признаки без лагов!"""
    X, y = [], []
    for chunk_id, group in df.groupby('chunk_full_id'):
        features = group[feature_cols].values
        targets = group['signal_barrier'].values
        targets = np.where(targets == 1, 1, 0) # Sigmoid format

        # Исправленный сдвиг: таргет берется от последней свечи в окне
        for i in range(len(group) - seq_length + 1):
            X.append(features[i : i + seq_length])
            y.append(targets[i + seq_length - 1])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# --- 4. Метрики и Обучение Моделей ---
def evaluate_predictions(y_true, y_pred, model_name=""):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f"[{model_name}] Accuracy: {acc:.4f} | F1: {f1:.4f}")
    return acc

def run_catboost(X_train, y_train, X_test, y_test, model_name):
    print(f"\n[{model_name}] Обучение (CUDA)...")
    model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0, task_type='GPU')
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    preds = model.predict(X_test)
    return evaluate_predictions(y_test, preds, model_name)

def run_keras_dense(X_train, y_train, X_test, y_test, model_name, is_modern=False):
    print(f"\n[{model_name}] Обучение (CUDA)...")
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    y_train_mapped = np.where(y_train == 1, 1, 0)
    y_test_mapped = np.where(y_test == 1, 1, 0)

    if is_modern: # Modern Tabular DNN
        model = Sequential([
            Input(shape=(X_train_s.shape[1],)),
            Dense(256, activation='relu'), BatchNormalization(), Dropout(0.3),
            Dense(128, activation='relu'), BatchNormalization(), Dropout(0.2),
            Dense(64, activation='relu'), BatchNormalization(), Dropout(0.1),
            Dense(1, activation='sigmoid')
        ])
    else: # Classic Deep MLP
        model = Sequential([
            Input(shape=(X_train_s.shape[1],)),
            Dense(256, activation='relu'),
            Dense(128, activation='relu'),
            Dense(64, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(X_train_s, y_train_mapped, validation_data=(X_test_s, y_test_mapped), epochs=100, batch_size=1024, callbacks=[early_stop], verbose=0)

    preds = np.where((model.predict(X_test_s, verbose=0) > 0.5).astype(int).flatten() == 1, 1, -1)
    return evaluate_predictions(y_test, preds, model_name)

def run_lstm(X_train_3d, y_train_3d, X_test_3d, y_test_3d, model_name):
    print(f"\n[{model_name}] Обучение (CUDA)...")
    samples, timesteps, features = X_train_3d.shape
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_3d.reshape(-1, features)).reshape(samples, timesteps, features)
    X_test_s = scaler.transform(X_test_3d.reshape(-1, features)).reshape(X_test_3d.shape[0], timesteps, features)

    # Упрощенная архитектура для защиты от переобучения на шуме
    model = Sequential([
        Input(shape=(timesteps, features)),
        LSTM(16, return_sequences=False),
        Dropout(0.4),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    model.fit(X_train_s, y_train_3d, validation_data=(X_test_s, y_test_3d), epochs=50, batch_size=512, callbacks=[early_stop], verbose=0)

    y_test_mapped = np.where(y_test_3d == 1, 1, -1)
    preds = np.where((model.predict(X_test_s, verbose=0) > 0.5).astype(int).flatten() == 1, 1, -1)
    return evaluate_predictions(y_test_mapped, preds, model_name)

def run_transformer(X_train_3d, y_train_3d, X_test_3d, y_test_3d, model_name):
    print(f"\n[{model_name}] Обучение (CUDA)...")
    samples, timesteps, features = X_train_3d.shape
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_3d.reshape(-1, features)).reshape(samples, timesteps, features)
    X_test_s = scaler.transform(X_test_3d.reshape(-1, features)).reshape(X_test_3d.shape[0], timesteps, features)

    inputs = Input(shape=(timesteps, features))
    attn_out = MultiHeadAttention(num_heads=2, key_dim=16)(inputs, inputs) # Упрощено для малых данных
    attn_out = Dropout(0.4)(attn_out)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn_out)

    ff_out = Dense(32, activation="relu")(out1)
    ff_out = Dropout(0.4)(ff_out)
    ff_out = Dense(features)(ff_out)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ff_out)

    x = GlobalAveragePooling1D()(out2)
    x = Dense(16, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss="binary_crossentropy", metrics=["accuracy"])
    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    model.fit(X_train_s, y_train_3d, validation_data=(X_test_s, y_test_3d), epochs=50, batch_size=512, callbacks=[early_stop], verbose=0)

    y_test_mapped = np.where(y_test_3d == 1, 1, -1)
    preds = np.where((model.predict(X_test_s, verbose=0) > 0.5).astype(int).flatten() == 1, 1, -1)
    return evaluate_predictions(y_test_mapped, preds, model_name)

# --- ОСНОВНОЙ ПАЙПЛАЙН ---
def main():
    if os.path.exists(ZIP_FILE): unpack_dataset(ZIP_FILE, OUTPUT_DIR)

    files = glob.glob(os.path.join(OUTPUT_DIR, '*.csv'))
    if not files: return print("CSV файлы не найдены.")

    print(f"\nЗагрузка {len(files)} CSV файлов...")
    df_list = []
    for f in tqdm(files, desc="Чтение CSV"):
        with open(f, 'r') as file: sep = ';' if ';' in file.readline() else ','
        temp_df = pd.read_csv(f, sep=sep)
        if 'symbol' not in temp_df.columns: temp_df['symbol'] = os.path.basename(f).replace('.csv', '')
        df_list.append(temp_df)

    df_raw = pd.concat(df_list, ignore_index=True)

    print("\nПодготовка данных...")
    df_with_chunks = get_contiguous_chunks(df_raw, freq_ms=FREQ_MS)
    df_filtered = filter_and_report_chunks(df_with_chunks, min_len=MIN_CHUNK_LENGTH)

    df_feat, all_features, base_3d_feats = create_financial_features(df_filtered, m_window=M)

    # Фильтры признаков для 2D
    features_full = all_features.copy()
    features_no_rd = [f for f in all_features if 'rd_' not in f]

    # Time-Based Сплит (2D Данные)
    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)
    split_idx = int(len(df_feat) * 0.8)
    train_data_2d, test_data_2d = df_feat.iloc[:split_idx], df_feat.iloc[split_idx:]

    y_train_2d = train_data_2d['signal_barrier'].values
    y_test_2d = test_data_2d['signal_barrier'].values

    # Генерация 3D Тензоров для LSTM/Transformer
    # ВАЖНО: Мы подаем ТОЛЬКО base_3d_feats (без лагов и без rd_value)
    print("\nПодготовка 3D тензоров для Sequence Models (Без лагов)...")
    X_train_3d, y_train_3d = create_3d_sequences(train_data_2d, base_3d_feats, seq_length=SEQ_LENGTH)
    X_test_3d, y_test_3d = create_3d_sequences(test_data_2d, base_3d_feats, seq_length=SEQ_LENGTH)

    print("\n" + "="*50)
    print("🚀 ЭКСПЕРИМЕНТЫ (ВСЕ МОДЕЛИ НА CUDA)")
    print("="*50)

    # 1. Бейзлайн EMA
    acc_ema = evaluate_predictions(y_test_2d, test_data_2d['rd_ema_cross_signal'].values, "1. Baseline EMA (No ML)")

    # 2. CatBoost С СИГНАЛОМ
    acc_cb_rd = run_catboost(train_data_2d[features_full], y_train_2d, test_data_2d[features_full], y_test_2d, "2. CatBoost [WITH rd_value]")

    # 3. CatBoost БЕЗ сигнала
    acc_cb_no_rd = run_catboost(train_data_2d[features_no_rd], y_train_2d, test_data_2d[features_no_rd], y_test_2d, "3. CatBoost [No rd_value]")

    # 4. Deep MLP
    acc_mlp = run_keras_dense(train_data_2d[features_no_rd], y_train_2d, test_data_2d[features_no_rd], y_test_2d, "4. Deep MLP [No rd_value]", is_modern=False)

    # 5. Tabular DNN
    acc_tab = run_keras_dense(train_data_2d[features_no_rd], y_train_2d, test_data_2d[features_no_rd], y_test_2d, "5. Tabular DNN [No rd_value]", is_modern=True)

    # 6. LSTM (3D)
    acc_lstm = run_lstm(X_train_3d, y_train_3d, X_test_3d, y_test_3d, "6. LSTM [No rd_value] (Clean Sequences)")

    # 7. Transformer (3D)
    acc_trans = run_transformer(X_train_3d, y_train_3d, X_test_3d, y_test_3d, "7. Transformer [No rd_value] (Clean Sequences)")

    print("\n================ ФИНАЛЬНЫЙ ЛИДЕРБОРД ================")
    print(f"Baseline EMA:                {acc_ema*100:.2f}%")
    print(f"CatBoost (С сигналом RD):    {acc_cb_rd*100:.2f}%")
    print("--------------------------------------------------")
    print("Битва на голом Price Action (БЕЗ rd_value):")
    print(f"CatBoost (Flat Lags):        {acc_cb_no_rd*100:.2f}%")
    print(f"Deep MLP (Flat Lags):        {acc_mlp*100:.2f}%")
    print(f"Tabular DNN (Flat Lags):     {acc_tab*100:.2f}%")
    print(f"LSTM (3D Sequences):         {acc_lstm*100:.2f}%")
    print(f"Transformer (3D Sequences):  {acc_trans*100:.2f}%")

if __name__ == "__main__":
    main()

✅ CUDA обнаружена! Доступно GPU: 1

Распаковка архива dataset_rework.zip...


Распаковка: 100%|██████████| 1710/1710 [00:00<00:00, 5461.73it/s]



Загрузка 833 CSV файлов...


Чтение CSV: 100%|██████████| 833/833 [00:01<00:00, 502.06it/s]



Подготовка данных...
Генерация исторических лагов (для 2D моделей)...


Лаги: 100%|██████████| 3/3 [00:00<00:00,  5.85it/s]



Подготовка 3D тензоров для Sequence Models (Без лагов)...

🚀 ЭКСПЕРИМЕНТЫ (ВСЕ МОДЕЛИ НА CUDA)
[1. Baseline EMA (No ML)] Accuracy: 0.4962 | F1: 0.4962

[2. CatBoost [WITH rd_value]] Обучение (CUDA)...
[2. CatBoost [WITH rd_value]] Accuracy: 0.9971 | F1: 0.9971

[3. CatBoost [No rd_value]] Обучение (CUDA)...
[3. CatBoost [No rd_value]] Accuracy: 0.8527 | F1: 0.8526

[4. Deep MLP [No rd_value]] Обучение (CUDA)...
[4. Deep MLP [No rd_value]] Accuracy: 0.8471 | F1: 0.8471

[5. Tabular DNN [No rd_value]] Обучение (CUDA)...
[5. Tabular DNN [No rd_value]] Accuracy: 0.8509 | F1: 0.8509

[6. LSTM [No rd_value] (Clean Sequences)] Обучение (CUDA)...
[6. LSTM [No rd_value] (Clean Sequences)] Accuracy: 0.8625 | F1: 0.8625

[7. Transformer [No rd_value] (Clean Sequences)] Обучение (CUDA)...
[7. Transformer [No rd_value] (Clean Sequences)] Accuracy: 0.6258 | F1: 0.6148

================ ФИНАЛЬНЫЙ ЛИДЕРБОРД ================
🔹 Baseline EMA:                49.62%
🏆 CatBoost (С сигналом RD):    99.71%
-